In [ ]:

import os, json, requests, zipfile, tarfile, io
from huggingface_hub import HfApi, hf_hub_download
OUT = "/kaggle/working/fl"; os.makedirs(OUT, exist_ok=True)
REPO = "onnx-community/Florence-2-base-ft"
api = HfApi()
info = api.model_info(REPO, files_metadata=True)
files = {s.rfilename: s.size for s in info.siblings}
print(json.dumps(files, indent=1))
parts = ["vision_encoder", "embed_tokens", "encoder_model", "decoder_model_merged"]
# choose the smallest good-quality variant per part: q4 > quantized(int8) > fp16
chosen = {}
for p in parts:
    for suf in ["_q4", "_quantized", "_int8", "_fp16", ""]:
        f = f"onnx/{p}{suf}.onnx"
        if f in files: chosen[p] = f; break
print("CHOSEN", chosen, sum(files[f] for f in chosen.values()) / 1e6, "MB")
want = [f for f in files if f.endswith(".json") and "/" not in f] + list(chosen.values())
for f in want:
    hf_hub_download(REPO, f, local_dir=OUT)
json.dump({"chosen": chosen, "sizes": {f: files[f] for f in want}}, open(OUT + "/manifest.json", "w"), indent=1)
# browser library + its runtime (exact matching versions), straight from the npm registry
def npm(pkg, ver=None):
    meta = requests.get(f"https://registry.npmjs.org/{pkg.replace('/', '%2f')}", timeout=60).json()
    v = ver or meta["dist-tags"]["latest"]; d = meta["versions"][v]
    tb = requests.get(d["dist"]["tarball"], timeout=300).content
    open(f"{OUT}/{pkg.replace('/', '_').replace('@', '')}-{v}.tgz", "wb").write(tb)
    return v, d.get("dependencies", {})
tv, deps = npm("@huggingface/transformers")
print("transformers.js", tv, deps)
ov = deps.get("onnxruntime-web", "").lstrip("^~")
print("ort-web", npm("onnxruntime-web", ov)[0])
os.system(f"cd /kaggle/working && zip -qr florence.zip fl && ls -la /kaggle/working")
